# Modelation mode choice

## 00_ charging and review data

In [19]:
import pandas as pd
import numpy as np
import sklearn
import statsmodels
import scipy
import matplotlib
import seaborn
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Statsmodels:", statsmodels.__version__)
print("SciPy:", scipy.__version__)

Pandas: 2.3.3
NumPy: 2.5.4
Scikit-learn: 1.9.1
Statsmodels: 0.15.0
SciPy: 1.18.1


In [10]:

from pathlib import Path

# Inspect the current working directory
project_path = Path.cwd()

print(f"Current working directory: {project_path}")
print(f"df_base already loaded: {'df_base' in globals()}")
print(f"df_model already loaded: {'df_model' in globals()}")

# Check expected analytical data files
candidate_paths = [
    project_path / "data" / "analytical",
    project_path.parent / "data" / "analytical",
    project_path / "508_TP_Proyect" / "data" / "analytical"
]

print("\nAvailable analytical data directories:")

for path in candidate_paths:
    print(f"{path}: {'Exists' if path.exists() else 'Not found'}")


Current working directory: d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\notebooks
df_base already loaded: True
df_model already loaded: False

Available analytical data directories:
d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\notebooks\data\analytical: Not found
d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\data\analytical: Exists
d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\notebooks\508_TP_Proyect\data\analytical: Not found


In [12]:
data_base_path = Path(
    r"D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\data\analytical\df_base.csv")
data_model_path = Path(
    r"D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\data\analytical\df_model.csv")

df_base = pd.read_csv(data_base_path, low_memory=False)
df_model = pd.read_csv(data_model_path, low_memory=False)

In [8]:
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100143 entries, 0 to 100142
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   key_viaje                     100143 non-null  object 
 1   modo_principal_agrupado       100143 non-null  object 
 2   modo_principal_desagrupado    100143 non-null  object 
 3   edad                          100143 non-null  int64  
 4   sexo                          100143 non-null  object 
 5   income_group                  100143 non-null  object 
 6   estra_hg                      100143 non-null  object 
 7   vehicle_availability          100143 non-null  object 
 8   has_car_license               100143 non-null  int64  
 9   has_motorcycle_license        100143 non-null  int64  
 10  has_public_transport_license  100143 non-null  int64  
 11  purpose_group                 100143 non-null  object 
 12  time_period_6                 100143 non-nul

In [13]:
df_base.head(5)

,key_viaje,modo_principal_agrupado,modo_principal_desagrupado,edad,sexo,income_group,estra_hg,vehicle_availability,has_car_license,has_motorcycle_license,has_public_transport_license,purpose_group,time_period_6,zat_centroid_distance_m,travel_frequency_3,education_group,occupation_group,household_size_group,duracion_min
0,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,AUTO,Vehículo privado como conductor,41,Hombre,NS_NR,3,2+,1,0,0,WORK,AM_PEAK,12730.630894,DAILY,SECONDARY,EMPLOYEE,4_5_PERSONS,60.0
1,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,AUTO,Vehículo privado como conductor,41,Hombre,NS_NR,3,2+,1,0,0,HOME_RETURN,PM_PEAK,12730.630894,DAILY,SECONDARY,EMPLOYEE,4_5_PERSONS,95.0
2,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,BICICLETA,Bicicleta convencional como conductor,42,Mujer,NS_NR,3,2+,0,0,0,WORK,AM_PEAK,1454.544478,DAILY,TECHNICAL_TECHNOLOGICAL,EMPLOYEE,4_5_PERSONS,30.0
3,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,BICICLETA,Bicicleta convencional como conductor,42,Mujer,NS_NR,3,2+,0,0,0,HOME_RETURN,PM_OFF_PEAK,1454.544478,DAILY,TECHNICAL_TECHNOLOGICAL,EMPLOYEE,4_5_PERSONS,3.0
4,uuid:0001256a-c2fc-4816-91c7-ab0348c4e74b/acce...,A PIE > 15 MIN,A pie (Mayores de 15 min),42,Mujer,NS_NR,3,2+,0,0,0,PERSONAL_ERRANDS,PM_OFF_PEAK,1134.070694,RARELY,TECHNICAL_TECHNOLOGICAL,EMPLOYEE,4_5_PERSONS,15.0


In [14]:

# Load the analytical dataset if it is not already in memory
if "df_model" not in globals():
    df_model = pd.read_csv(data_model_path, low_memory=False)

# Recover the individual identifier associated with each trip
person_map = (
    df_model[["key_viaje", "key_pers"]]
    .drop_duplicates()
)

# Validate the trip-to-person relationship
if person_map["key_viaje"].duplicated().any():
    raise ValueError("Some trips are associated with multiple individuals.")

# Merge individual identifiers into the modeling dataset
if "key_pers" not in df_base.columns:
    df_base = df_base.merge(
        person_map,
        on="key_viaje",
        how="left",
        validate="many_to_one"
    )

# Report recovery results
print(f"Modeling observations: {len(df_base):,}")
print(f"Unique individuals: {df_base['key_pers'].nunique():,}")
print(f"Missing individual identifiers: {df_base['key_pers'].isna().sum():,}")
print(f"Target categories: {df_base['modo_principal_agrupado'].nunique()}")


Modeling observations: 100,143
Unique individuals: 48,528
Missing individual identifiers: 0
Target categories: 11


## 01 Base_modeling

In [15]:

# Define target variable
target = "modo_principal_agrupado"

# Define baseline explanatory variables
baseline_features = [
    "edad",
    "sexo",
    "income_group",
    "estra_hg",
    "vehicle_availability",
    "has_car_license",
    "has_motorcycle_license",
    "has_public_transport_license",
    "purpose_group",
    "time_period_6",
    "zat_centroid_distance_m"
]

# Separate predictors and target
X = df_base[baseline_features].copy()
y = df_base[target].copy()

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [
    col for col in baseline_features
    if col not in numerical_features
]

# Basic validation
assert X.isna().sum().sum() == 0, "Missing values detected in predictors."
assert y.notna().all(), "Missing values detected in target."

print(f"Observations: {len(X):,}")
print(f"Predictors: {len(baseline_features)}")
print(f"Numerical features: {numerical_features}")
print(f"Categorical features: {categorical_features}")
print(f"Target classes: {y.nunique()}")
print("\nTarget distribution:")
print(y.value_counts())


Observations: 100,143
Predictors: 11
Numerical features: ['edad', 'has_car_license', 'has_motorcycle_license', 'has_public_transport_license', 'zat_centroid_distance_m']
Categorical features: ['sexo', 'income_group', 'estra_hg', 'vehicle_availability', 'purpose_group', 'time_period_6']
Target classes: 11

Target distribution:
modo_principal_agrupado
TRANSPORTE PÚBLICO    29400
A PIE > 15 MIN        24923
AUTO                  12582
A PIE <15 MIN         12387
MOTO                   6247
BICICLETA              6101
TAXI OCUPADO           3266
TRANSPORTE ESCOLAR     2058
ESPECIAL OCUPADO       1981
INFORMAL                866
OTRO                    332
Name: count, dtype: int64


### 1.1 Particion del data base

In [ ]:
# Define reproducibility parameters
RANDOM_STATE = 42
N_SPLITS = 5

# Define grouping variable
groups = df_base["key_pers"]

# Create grouped and stratified holdout split
outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

train_idx, test_idx = next( outer_cv.split(X, y, groups=groups))

# Create development and test datasets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# Validate individual separation
assert set(groups_train).isdisjoint(set(groups_test))

# Report partition statistics
print("PARTITION SUMMARY")
print("-" * 40)

print(f"Training observations: {len(X_train):,} "
      f"({len(X_train)/len(X):.2%})")

print(f"Test observations: {len(X_test):,} "
      f"({len(X_test)/len(X):.2%})")

print(f"Training individuals: {groups_train.nunique():,}")
print(f"Test individuals: {groups_test.nunique():,}")

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))

# Define grouped cross-validation for model development
inner_cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


PARTITION SUMMARY
----------------------------------------
Training observations: 80,114 (80.00%)
Test observations: 20,029 (20.00%)
Training individuals: 38,821
Test individuals: 9,707

Training target distribution:
modo_principal_agrupado
TRANSPORTE PÚBLICO    0.294
A PIE > 15 MIN        0.249
AUTO                  0.126
A PIE <15 MIN         0.124
MOTO                  0.062
BICICLETA             0.061
TAXI OCUPADO          0.033
TRANSPORTE ESCOLAR    0.021
ESPECIAL OCUPADO      0.020
INFORMAL              0.009
OTRO                  0.003
Name: proportion, dtype: float64

Test target distribution:
modo_principal_agrupado
TRANSPORTE PÚBLICO    0.294
A PIE > 15 MIN        0.249
AUTO                  0.126
A PIE <15 MIN         0.124
MOTO                  0.062
BICICLETA             0.061
TAXI OCUPADO          0.033
TRANSPORTE ESCOLAR    0.021
ESPECIAL OCUPADO      0.020
INFORMAL              0.009
OTRO                  0.003
Name: proportion, dtype: float64


### 1.2 Preprocesamiento

In [21]:

# Identify binary and categorical variables using training data only

binary_features = []
categorical_features = []
numerical_features = []

for col in baseline_features:
    unique_values = X_train[col].dropna().unique()

    if len(unique_values) == 2:
        binary_features.append(col)

    elif (
        pd.api.types.is_numeric_dtype(X_train[col])
    ):
        numerical_features.append(col)

    else:
        categorical_features.append(col)

# Display variable classification
print("BINARY FEATURES")
for col in binary_features:
    print(f"{col}: {sorted(X_train[col].dropna().unique().tolist(), key=str)}")

print("\nCATEGORICAL FEATURES")
for col in categorical_features:
    print(f"{col}: {X_train[col].nunique()} categories")

print("\nNUMERICAL FEATURES")
for col in numerical_features:
    print(f"{col}: {X_train[col].dtype}")

# Validate that all predictors are classified exactly once
classified_features = (
    binary_features + categorical_features + numerical_features
)

assert len(classified_features) == len(baseline_features)
assert len(set(classified_features)) == len(baseline_features)

print("\nFeature classification validated.")


BINARY FEATURES
has_car_license: [0, 1]
has_motorcycle_license: [0, 1]
has_public_transport_license: [0, 1]

CATEGORICAL FEATURES
sexo: 3 categories
income_group: 4 categories
estra_hg: 7 categories
vehicle_availability: 3 categories
purpose_group: 8 categories
time_period_6: 6 categories

NUMERICAL FEATURES
edad: int64
zat_centroid_distance_m: float64

Feature classification validated.


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("binary", "passthrough", binary_features),
        ("categorical", OneHotEncoder(
            drop="first",
            handle_unknown="ignore"
        ), categorical_features),
        ("numerical", StandardScaler(), numerical_features)
    ],
    remainder="drop"
)

print("Preprocessor configured successfully.")

Preprocessor configured successfully.


In [24]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Original features:", X_train.shape[1])
print("Processed features:", X_train_processed.shape[1])
print("Processed matrix shape:", X_train_processed.shape)

Original features: 11
Processed features: 30
Processed matrix shape: (80114, 30)


In [26]:
feature_names = preprocessor.get_feature_names_out()

print("Processed feature names:")
for i, feature in enumerate(feature_names, start=1):
    print(f"{i:02d}. {feature}")

print(f"\nTotal processed features: {len(feature_names)}")

Processed feature names:
01. binary__has_car_license
02. binary__has_motorcycle_license
03. binary__has_public_transport_license
04. categorical__sexo_Intersexual
05. categorical__sexo_Mujer
06. categorical__income_group_LOW_INCOME
07. categorical__income_group_MEDIUM_INCOME
08. categorical__income_group_NS_NR
09. categorical__estra_hg_2
10. categorical__estra_hg_3
11. categorical__estra_hg_4
12. categorical__estra_hg_5
13. categorical__estra_hg_6
14. categorical__estra_hg_No aplica
15. categorical__vehicle_availability_1
16. categorical__vehicle_availability_2+
17. categorical__purpose_group_ESCORT
18. categorical__purpose_group_HEALTH
19. categorical__purpose_group_HOME_RETURN
20. categorical__purpose_group_LEISURE
21. categorical__purpose_group_PERSONAL_ERRANDS
22. categorical__purpose_group_SOCIAL_RELIGIOUS
23. categorical__purpose_group_WORK
24. categorical__time_period_6_AM_PEAK
25. categorical__time_period_6_EVENING
26. categorical__time_period_6_OVERNIGHT
27. categorical__time_

In [27]:
# Identify reference categories used by OneHotEncoder

encoder = preprocessor.named_transformers_["categorical"]

for feature, categories, drop_idx in zip(
    categorical_features,
    encoder.categories_,
    encoder.drop_idx_
):
    reference = categories[drop_idx] if drop_idx is not None else None
    print(f"{feature}: {reference}")

sexo: Hombre
income_group: HIGH_INCOME
estra_hg: 1
vehicle_availability: 0
purpose_group: EDUCATION
time_period_6: AM_OFF_PEAK


In [29]:
import statsmodels.api as sm
from scipy.sparse import issparse

# Define AUTO as the reference alternative
mode_categories = ["AUTO"] + sorted(
    [mode for mode in y_train.unique() if mode != "AUTO"]
)

y_train_mnl = pd.Series(
    pd.Categorical(
        y_train,
        categories=mode_categories
    ).codes,
    index=y_train.index,
    name="mode_choice"
)

# Transform training predictors
X_train_mnl = preprocessor.fit_transform(X_train)

if issparse(X_train_mnl):
    X_train_mnl = X_train_mnl.toarray()

X_train_mnl = pd.DataFrame(
    X_train_mnl,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

# Add intercept
X_train_mnl = sm.add_constant(X_train_mnl, has_constant="add")

# Verify model inputs
print("Alternative coding:")
for code, mode in enumerate(mode_categories):
    print(f"{code}: {mode}")

print("\nDesign matrix:", X_train_mnl.shape)
print("Target vector:", y_train_mnl.shape)
print("Reference alternative:", mode_categories[0])

Alternative coding:
0: AUTO
1: A PIE <15 MIN
2: A PIE > 15 MIN
3: BICICLETA
4: ESPECIAL OCUPADO
5: INFORMAL
6: MOTO
7: OTRO
8: TAXI OCUPADO
9: TRANSPORTE ESCOLAR
10: TRANSPORTE PÚBLICO

Design matrix: (80114, 31)
Target vector: (80114,)
Reference alternative: AUTO


In [ ]:
# Estimate the baseline Multinomial Logit model

mnl_model = sm.MNLogit(
    y_train_mnl,
    X_train_mnl
)

mnl_results = mnl_model.fit(
    method="bfgs",
    maxiter=200,
    gtol=1e-6,
    disp=True
)


d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\.venv\Lib\site-packages\scipy\optimize\_optimize.py:1330: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)


         Current function value: 1.353112
         Iterations: 200
         Function evaluations: 201
         Gradient evaluations: 201


d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:858: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mnfit = base.LikelihoodModel.fit(


In [31]:
# Inspect MNL optimization diagnostics

print("Optimization diagnostics:")
for key, value in mnl_results.mle_retvals.items():
    print(f"{key}: {value}")

print("\nParameter checks:")
print("Parameters estimated:", mnl_results.params.size)
print("All parameters finite:", np.isfinite(mnl_results.params).all())
print("Log-likelihood:", mnl_results.llf)

Optimization diagnostics:
fopt: 1.3531117389228056
gopt: [-1.37118314e-05 -7.01165725e-05  9.59258664e-06  1.61973377e-05
  1.25992389e-05  1.88032242e-05 -2.26075128e-05 -1.30938810e-04
  8.90649746e-05 -7.87871623e-05  3.42318863e-07 -8.47616402e-05
  8.44723172e-05 -6.61686000e-05  5.97599701e-05  2.16969357e-04
 -6.03175502e-05 -2.61338578e-05 -2.15078706e-05  1.27404824e-04
 -6.97371551e-05 -5.63451386e-05 -6.65177547e-05  3.53132705e-05
 -2.37457886e-05 -7.96715567e-05  7.45812482e-07  3.14847062e-05
  1.67146697e-05  2.51396730e-06 -8.26742095e-05 -1.85961303e-04
 -2.66805238e-04 -4.57185108e-05 -2.03623365e-05  2.29739905e-05
  1.44960439e-04  2.21445316e-05  1.72098063e-04 -4.80371721e-04
  1.41946508e-04 -3.52629877e-04 -7.65490291e-05  1.36290715e-05
 -4.99329386e-05  9.03054578e-05 -2.69530595e-04  6.43660625e-06
  5.27704656e-05  7.21311412e-06 -1.82871712e-04  3.10042286e-05
 -2.34991737e-06 -3.82601951e-06 -1.48180561e-04  2.94664661e-05
  1.89506587e-05  1.41722749e-05 

In [32]:
# Continue MNL optimization from previous estimates

mnl_results_2 = mnl_model.fit(
    start_params=mnl_results.params.to_numpy().ravel(order="F"),
    method="bfgs",
    maxiter=1000,
    gtol=1e-6,
    disp=True
)

print("\nConvergence:", mnl_results_2.mle_retvals["converged"])
print("Iterations:", mnl_results_2.mle_retvals["fcalls"])
print("Log-likelihood:", mnl_results_2.llf)

d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\.venv\Lib\site-packages\scipy\optimize\_optimize.py:1330: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)


         Current function value: 1.351974
         Iterations: 1000
         Function evaluations: 1002
         Gradient evaluations: 1002

Convergence: False
Iterations: 1002
Log-likelihood: -108312.02034383279


d:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:858: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mnfit = base.LikelihoodModel.fit(


In [33]:
# Estimate MNL using L-BFGS optimization

mnl_results_lbfgs = mnl_model.fit(
    method="lbfgs",
    maxiter=1000,
    pgtol=1e-6,
    disp=True
)

print("\nConvergence:", mnl_results_lbfgs.mle_retvals["converged"])
print("Iterations:", mnl_results_lbfgs.mle_retvals["fcalls"])
print("Log-likelihood:", mnl_results_lbfgs.llf)


Convergence: True
Iterations: 943
Log-likelihood: -108311.7273193966


In [34]:
# Evaluate MNL estimation diagnostics

print("CONVERGENCE:", mnl_results_lbfgs.mle_retvals["converged"])
print("OPTIMIZER:", mnl_results_lbfgs.mle_settings["optimizer"])

print("\nOPTIMIZATION DETAILS")
for key in ["nit", "fcalls", "warnflag", "fopt"]:
    print(f"{key}: {mnl_results_lbfgs.mle_retvals.get(key, 'Not available')}")

print("\nMODEL FIT")
print(f"Log-likelihood: {mnl_results_lbfgs.llf:.4f}")
print(f"AIC: {mnl_results_lbfgs.aic:.4f}")
print(f"BIC: {mnl_results_lbfgs.bic:.4f}")

print("\nPARAMETER DIAGNOSTICS")
print("Finite coefficients:", np.isfinite(mnl_results_lbfgs.params.to_numpy()).all())
print("Finite standard errors:", np.isfinite(mnl_results_lbfgs.bse.to_numpy()).all())

CONVERGENCE: True
OPTIMIZER: lbfgs

OPTIMIZATION DETAILS
nit: Not available
fcalls: 943
warnflag: 0
fopt: 1.3519700341937313

MODEL FIT
Log-likelihood: -108311.7273
AIC: 217243.4546
BIC: 220123.7285

PARAMETER DIAGNOSTICS
Finite coefficients: True
Finite standard errors: True


In [35]:
# Build the MNL coefficient results table

params = mnl_results_lbfgs.params.copy()
standard_errors = mnl_results_lbfgs.bse.copy()
p_values = mnl_results_lbfgs.pvalues.copy()

params.columns = mode_categories[1:]
standard_errors.columns = mode_categories[1:]
p_values.columns = mode_categories[1:]

coef_table = pd.concat({
    "Coefficient": params.stack(),
    "Std_Error": standard_errors.stack(),
    "P_value": p_values.stack()
}, axis=1).reset_index()

coef_table.columns = [
    "Feature", "Alternative",
    "Coefficient", "Std_Error", "P_value"
]

coef_table["Z_stat"] = (
    coef_table["Coefficient"] / coef_table["Std_Error"]
)

coef_table["CI_lower"] = (
    coef_table["Coefficient"] - 1.96 * coef_table["Std_Error"]
)

coef_table["CI_upper"] = (
    coef_table["Coefficient"] + 1.96 * coef_table["Std_Error"]
)

coef_table["Significance"] = np.select(
    [
        coef_table["P_value"] < 0.01,
        coef_table["P_value"] < 0.05,
        coef_table["P_value"] < 0.10
    ],
    ["***", "**", "*"],
    default=""
)

coef_table.head(15)

,Feature,Alternative,Coefficient,Std_Error,P_value,Z_stat,CI_lower,CI_upper,Significance
0,const,A PIE <15 MIN,2.706292,0.131045,9.425398e-95,20.651695,2.449445,2.963140,***
1,const,A PIE > 15 MIN,3.961519,0.121915,1.287952e-231,32.494193,3.722566,4.200472,***
2,const,BICICLETA,-0.628024,0.160565,9.178637e-05,-3.911338,-0.942732,-0.313317,***
3,const,ESPECIAL OCUPADO,-0.426299,0.225754,5.898166e-02,-1.888330,-0.868778,0.016180,*
4,const,INFORMAL,0.952630,0.263793,3.046937e-04,3.611276,0.435595,1.469664,***
5,const,MOTO,-1.724768,0.200027,6.540011e-18,-8.622689,-2.116821,-1.332716,***
6,const,OTRO,-2.451234,0.698464,4.490090e-04,-3.509465,-3.820223,-1.082245,***
7,const,TAXI OCUPADO,0.911031,0.199771,5.105961e-06,4.560386,0.519481,1.302582,***
8,const,TRANSPORTE ESCOLAR,-1.691810,0.243266,3.536342e-12,-6.954572,-2.168611,-1.215009,***
9,const,TRANSPORTE PÚBLICO,3.916757,0.119703,7.914628e-235,32.720760,3.682140,4.151374,***


In [37]:
# Test the joint significance of each predictor across alternatives

wald_results = []

for feature in X_train_mnl.columns:
    if feature == "const":
        continue

    feature_positions = [
        i for i, name in enumerate(X_train_mnl.columns)
        if name == feature
    ]

    restrictions = np.zeros(
        (len(feature_positions) * (len(mode_categories) - 1),
         len(mnl_results_lbfgs.params.to_numpy().ravel()))
    )

    # Joint Wald test will be constructed in the next step

In [38]:
# Inspect the MNL parameter structure

print("Parameter matrix shape:", mnl_results_lbfgs.params.shape)
print("Number of predictors:", len(mnl_results_lbfgs.params.index))
print("Alternatives:", mnl_results_lbfgs.params.columns.tolist())
print("First predictor names:", mnl_results_lbfgs.params.index[:5].tolist())

Parameter matrix shape: (31, 10)
Number of predictors: 31
Alternatives: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
First predictor names: ['const', 'binary__has_car_license', 'binary__has_motorcycle_license', 'binary__has_public_transport_license', 'categorical__sexo_Intersexual']


In [41]:
# Perform joint Wald tests for each predictor across alternatives

from scipy.stats import chi2

params = mnl_results_lbfgs.params
cov_matrix = mnl_results_lbfgs.cov_params().to_numpy()

n_predictors, n_alternatives = params.shape
parameter_vector = params.to_numpy().ravel(order="F")

wald_results = []

for feature in params.index:
    if feature == "const":
        continue

    feature_idx = params.index.get_loc(feature)

    positions = (
        feature_idx + n_predictors * np.arange(n_alternatives)
    )

    beta = parameter_vector[positions]
    covariance = cov_matrix[np.ix_(positions, positions)]

    wald_stat = beta @ np.linalg.pinv(covariance) @ beta
    degrees_freedom = n_alternatives
    p_value = chi2.sf(wald_stat, degrees_freedom)

    wald_results.append({
        "Feature": feature,
        "Wald_Statistic": wald_stat,
        "Degrees_Freedom": degrees_freedom,
        "P_value": p_value
    })

wald_table = pd.DataFrame(wald_results)

wald_table["Significance"] = np.select(
    [
        wald_table["P_value"] < 0.01,
        wald_table["P_value"] < 0.05,
        wald_table["P_value"] < 0.10
    ],
    ["***", "**", "*"],
    default=""
)

wald_table = wald_table.sort_values("P_value")

display(wald_table)

,Feature,Wald_Statistic,Degrees_Freedom,P_value,Significance
0,binary__has_car_license,3601.599615,10,0.000000e+00,***
15,categorical__vehicle_availability_2+,5106.939406,10,0.000000e+00,***
28,numerical__edad,1684.942942,10,0.000000e+00,***
4,categorical__sexo_Mujer,1955.985079,10,0.000000e+00,***
14,categorical__vehicle_availability_1,4880.846458,10,0.000000e+00,***
29,numerical__zat_centroid_distance_m,10449.431543,10,0.000000e+00,***
1,binary__has_motorcycle_license,6665.134666,10,0.000000e+00,***
22,categorical__purpose_group_WORK,1449.375244,10,2.162506e-305,***
12,categorical__estra_hg_6,1120.009076,10,2.563196e-234,***
13,categorical__estra_hg_No aplica,1084.411928,10,1.209476e-226,***
